In [8]:
import csv
import json
import math
import statistics
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple
from datetime import date, datetime

In [9]:
def to_int(value: Any) -> Optional[int]:
    if value is None or value == "":
        return None
    try:
        return int(float(value)) if isinstance(value, str) else int(value)
    except (ValueError, TypeError):
        return None


def load_csv(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        return [dict(row) for row in reader]


def parse_datetime(value: Any) -> Optional[datetime]:
    if value is None or value == "":
        return None
    if isinstance(value, datetime):
        return value
    if isinstance(value, str):
        for fmt in (
            "%Y-%m-%d %H:%M:%S",
            "%Y/%m/%d %H:%M:%S",
            "%d/%m/%Y %H:%M:%S",
            "%Y-%m-%d",
            "%Y/%m/%d",
            "%d/%m/%Y",
        ):
            try:
                return datetime.strptime(value.strip(), fmt)
            except ValueError:
                continue
    return None


def filter_games_by_date(records: List[Dict[str, Any]], start: date, end: date) -> List[Dict[str, Any]]:
    filtered: List[Dict[str, Any]] = []
    for record in records:
        date_value = record.get("gameDateTimeEst") or record.get("gameDate") or record.get("dateTime")
        dt = parse_datetime(date_value)
        if dt is None:
            continue
        if start < dt.date() < end:
            filtered.append({
                "dateTime": dt.strftime("%Y-%m-%d %H:%M:%S"),
                "homeTeamName": record.get("hometeamName") or record.get("homeTeamName"),
                "awayTeamName": record.get("awayteamName") or record.get("awayTeamName"),
                "homeScore": to_int(record.get("homeScore")),
                "awayScore": to_int(record.get("awayScore")),
            })
    return filtered


def save_filtered_games_csv(path: str, records: List[Dict[str, Any]]) -> None:
    fieldnames = ["dateTime", "homeTeamName", "awayTeamName", "homeScore", "awayScore"]
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)


def process_games_csv(input_path: str, output_path: str) -> None:
    games = load_csv(input_path)
    filtered_games = filter_games_by_date(
        games,
        start=date(2025, 12, 31),
        end=date(2026, 4, 18),
    )
    save_filtered_games_csv(output_path, filtered_games)
    print(f"Processamento completo: {len(filtered_games)} jogos salvos em {output_path}")

In [15]:
resultado = process_games_csv("Games.csv", "Games_filtrado.csv")

Processamento completo: 747 jogos salvos em Games_filtrado.csv


In [2]:
import csv
import json
from pathlib import Path


def csv_to_json(csv_path: str, json_path: str) -> None:
    csv_file = Path(csv_path)
    json_file = Path(json_path)
    with csv_file.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        rows = [row for row in reader]

    with json_file.open("w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)


# Exemplo de uso:
# Ajuste os caminhos se necessário para o local do CSV e do JSON
csv_to_json("Games_filtrado.csv", "Games.json")
print("CSV convertido para JSON com sucesso.")


CSV convertido para JSON com sucesso.
